In [ ]:
from pathlib import Path
import os
import sys
import gzip
import json
import shutil
import hashlib
import importlib
import subprocess
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import scanpy as sc
import cuml
#gpu版umap
import cudf
import umap
#cpu版umap

from IPython.display import display
from scipy.sparse import load_npz
from sklearn.preprocessing import StandardScaler, normalize
from cuml.decomposition import PCA

# 如需使用 CPU 版本的 PCA：
# from sklearn.decomposition import PCA

pd.set_option('display.max_colwidth', None)   # 单元格内容不截断
pd.set_option('display.max_columns', None)    # 显示所有列
pd.set_option('display.width', None)          # 自动适配宽度
pd.set_option('display.max_rows', 200)        # 需要时显示更多行

pd.set_option('display.max_colwidth', None)   # 单元格内容不截断
pd.set_option('display.max_columns', None)    # 显示所有列
pd.set_option('display.width', None)          # 自动适配宽度
pd.set_option('display.max_rows', 200)        # 需要时显示更多行
# The second dataset is organized as old-cDNA/new-cDNA and old-beads/new-beads.
# Change this to 'old' to process the old-* FASTQ pair instead.
DATASET_VERSION = 'new'

In [ ]:
PROJECT_DIR = Path('/p1/zulab_users/jtian/my_jupyter/IRI-seq')
SCRIPT_FROM_GITHUB = PROJECT_DIR / 'script_from_github'
INPUT_DIR = Path('/p2/zulab/jtian/data/IRISeq/second/data/Data')
OUTPUT_ROOT = Path('/p2/zulab/jtian/data/IRISeq/second/output/IRIs-pipeline')
OUTPUT_DIR = OUTPUT_ROOT / DATASET_VERSION
REFERENCE_DIR = Path('/p2/zulab/jtian/data/IRISeq/reference')


CDNA_SAMPLE = f'{DATASET_VERSION}-cDNA'
BEAD_SAMPLE = f'{DATASET_VERSION}-beads'
SAMPLE_ID = DATASET_VERSION
OUTPUT_PREFIX = SAMPLE_ID

FASTQ_SOURCE_FILES = {
    CDNA_SAMPLE: {
        'run_accession': CDNA_SAMPLE,
        'description': f'second dataset {DATASET_VERSION} cDNA',
        'R1': INPUT_DIR / CDNA_SAMPLE / f'{CDNA_SAMPLE}_R1.fq.gz',
        'R2': INPUT_DIR / CDNA_SAMPLE / f'{CDNA_SAMPLE}_R2.fq.gz',
    },
    BEAD_SAMPLE: {
        'run_accession': BEAD_SAMPLE,
        'description': f'second dataset {DATASET_VERSION} bead interaction',
        'R1': INPUT_DIR / BEAD_SAMPLE / f'{BEAD_SAMPLE}_R1.fq.gz',
        'R2': INPUT_DIR / BEAD_SAMPLE / f'{BEAD_SAMPLE}_R2.fq.gz',
        'R3': INPUT_DIR / BEAD_SAMPLE / f'{BEAD_SAMPLE}_R2.fq.gz',
    },
}

# Optional processed files. The second raw-data folder currently does not contain these;
# validation sections will skip processed comparisons when they are absent.
PROCESSED_METADATA_CSV = INPUT_DIR / 'GSE270383_meta_data.csv.gz'
PROCESSED_CONNECTION_CSV = INPUT_DIR / f'{BEAD_SAMPLE}.spatial.csv.gz'
PROCESSED_COUNT_MTX = INPUT_DIR / 'GSE270383_count.mtx.gz'
PROCESSED_BARCODES_TSV = INPUT_DIR / 'GSE270383_barcodes.tsv.gz'
PROCESSED_GENES_TSV = INPUT_DIR / 'GSE270383_genes.tsv.gz'

PICKLE_DIR = SCRIPT_FROM_GITHUB / 'Bead_bc_white_list'
BARCODE_1_FILE = PICKLE_DIR / 'Spatial_R2_barcode_1.pickle'
BARCODE_2_FILE = PICKLE_DIR / 'Spatial_R2_barcode_2.pickle'
BARCODE_3_FILE = PICKLE_DIR / 'Spatial_R2_barcode_3.pickle'
BARCODE_4_FILE = PICKLE_DIR / 'Spatial_R2_barcode_4.pickle'
BARCODE_4_BEAD1_FILE = PICKLE_DIR / 'Spatial_R2_barcode_4_bead1.pickle'

STAR_INDEX = REFERENCE_DIR / 'STAR_mm10_GENCODE_M25'
GENE_REFERENCE_PICKLE = REFERENCE_DIR / 'Gene_annotation' / 'mm10_GENCODE_M25_Gene_reference.pickle'

STAR = shutil.which('STAR') or 'STAR'
SAMTOOLS = shutil.which('samtools') or 'samtools'
CUTADAPT = shutil.which('cutadapt') or 'cutadapt'

# Runtime controls. Keep force flags False for resumable reruns.
RUN_CDNA_PIPELINE = True
RUN_BEAD_PIPELINE = True
RUN_EXPRESSION_ANALYSIS = True
RUN_RECONSTRUCTION = True
RUN_VALIDATION = False
FORCE_RELINK_FASTQ = False
FORCE_RERUN_CDNA = False
FORCE_RERUN_BEADS = False
FORCE_RERUN_RECONSTRUCTION = False

SAMPLE_SHEET_DIR = OUTPUT_DIR / 'sample_sheets'
FASTQ_DIR = OUTPUT_DIR / 'prepared_fastq'
REPORT_DIR = OUTPUT_DIR / 'reports'
FIGURE_OUTPUT = OUTPUT_DIR / 'figures'
VALIDATION_OUTPUT = OUTPUT_DIR / 'validation'

CDNA_OUTPUT = OUTPUT_DIR / 'cDNA'
FASTQ_BARCODE_ATTACHED = CDNA_OUTPUT / 'Fastq_barcode_attached'
FASTQ_TRIMMED = CDNA_OUTPUT / 'Fastq_trimmed'
SAM_STAR = CDNA_OUTPUT / 'Sam_STAR'
SAM_FILTERED = CDNA_OUTPUT / 'Sam_filtered'
SAM_RMDUP = CDNA_OUTPUT / 'Sam_rmdup'
BED_GENE_COUNT = CDNA_OUTPUT / 'Bed_gene_count'
SUMMARY_GENE_COUNT = CDNA_OUTPUT / 'Summary_gene_count'
ADATA_FOLDER = CDNA_OUTPUT / 'Adata'
EXPRESSION_OUTPUT = CDNA_OUTPUT / 'expression_analysis'

BEAD_OUTPUT = OUTPUT_DIR / 'beads'
UMI_ATTACH = BEAD_OUTPUT / 'UMI_attach'
SPATIAL_BARCODE = BEAD_OUTPUT / 'Spatial_barcode_extraction'
DEDUPLICATE_SPATIAL = BEAD_OUTPUT / 'Spatial_barcode_rmdup'
BEAD_REPORT_FOLDER = BEAD_OUTPUT / 'report' / 'read_num_spatial_barcode'
CONNECTION_OUTPUT = OUTPUT_DIR / 'connections'

for directory in [
    OUTPUT_DIR, SAMPLE_SHEET_DIR, FASTQ_DIR, REPORT_DIR, FIGURE_OUTPUT, VALIDATION_OUTPUT,
    CDNA_OUTPUT, FASTQ_BARCODE_ATTACHED, FASTQ_TRIMMED, SAM_STAR, SAM_FILTERED, SAM_RMDUP,
    BED_GENE_COUNT, SUMMARY_GENE_COUNT, ADATA_FOLDER, EXPRESSION_OUTPUT,
    BEAD_OUTPUT, UMI_ATTACH, SPATIAL_BARCODE, DEDUPLICATE_SPATIAL, BEAD_REPORT_FOLDER,
    CONNECTION_OUTPUT,
]:
    directory.mkdir(parents=True, exist_ok=True)

print('DATASET_VERSION =', DATASET_VERSION)
print('INPUT_DIR       =', INPUT_DIR)
print('OUTPUT_ROOT     =', OUTPUT_ROOT)
print('OUTPUT_DIR      =', OUTPUT_DIR)
print('CDNA_SAMPLE     =', CDNA_SAMPLE)
print('BEAD_SAMPLE     =', BEAD_SAMPLE)
print('STAR            =', STAR)
print('SAMTOOLS        =', SAMTOOLS)
print('CUTADAPT        =', CUTADAPT)


## 2. Preflight checks

This cell verifies all expected inputs, author scripts, barcode whitelist files, and external command-line tools before starting the expensive steps.


In [ ]:
required_files = []
for sample, reads in FASTQ_SOURCE_FILES.items():
    for read_name, path in reads.items():
        if read_name in {'R1', 'R2'}:
            required_files.append((path, f'{sample} {read_name} FASTQ'))

required_files.extend([
    (BARCODE_1_FILE, 'barcode whitelist 1'),
    (BARCODE_2_FILE, 'barcode whitelist 2'),
    (BARCODE_3_FILE, 'barcode whitelist 3'),
    (BARCODE_4_FILE, 'barcode whitelist 4'),
    (BARCODE_4_BEAD1_FILE, 'barcode whitelist 4 bead1'),
    (STAR_INDEX, 'STAR mm10 index directory'),
    (GENE_REFERENCE_PICKLE, 'mm10 gene reference pickle'),
    (SCRIPT_FROM_GITHUB / 'EasySpatial' / 'Spatial_UMI_barcode_extraction.py', 'author cDNA barcode script'),
    (SCRIPT_FROM_GITHUB / 'Bead_interaction_pipeline' / 'UMI_barcode_extraction.py', 'author bead UMI script'),
    (SCRIPT_FROM_GITHUB / 'Sample_reconstruction_code_CPU.ipynb', 'author CPU reconstruction notebook'),
])

preflight_rows = []
for path, label in required_files:
    exists = Path(path).exists()
    preflight_rows.append({'label': label, 'path': str(path), 'required': True, 'exists': exists})
    if not exists:
        raise FileNotFoundError(f'Missing {label}: {path}')

if RUN_CDNA_PIPELINE:
    missing_tools = [tool for tool, resolved in {'STAR': STAR, 'samtools': SAMTOOLS, 'cutadapt': CUTADAPT}.items() if shutil.which(resolved) is None and not Path(resolved).exists()]
    if missing_tools:
        raise EnvironmentError(
            'Missing external tools required for full cDNA processing: ' + ', '.join(missing_tools) +
            '. Add them to PATH or set STAR/SAMTOOLS/CUTADAPT to absolute paths in the configuration cell.'
        )

python_package_checks = {
    'scanpy': RUN_EXPRESSION_ANALYSIS or RUN_VALIDATION,
    'umap': RUN_RECONSTRUCTION,
    'sklearn': RUN_RECONSTRUCTION or RUN_VALIDATION,
    'scipy': RUN_RECONSTRUCTION or RUN_VALIDATION,
    'matplotlib': True,
}
missing_packages = [pkg for pkg, needed in python_package_checks.items() if needed and importlib.util.find_spec(pkg) is None]
if missing_packages:
    raise ImportError(
        'Missing Python packages required by enabled sections: ' + ', '.join(missing_packages) +
        '. Install them in the active Jupyter kernel or set the relevant RUN_* flags to False.'
    )

display(pd.DataFrame(preflight_rows))
print('Preflight checks passed for required files and packages.')


## 3. Prepare FASTQ names and sample sheets

The author scripts expect files named as `{sample}.R1.fastq.gz`, `{sample}.R2.fastq.gz`, and for the interaction pipeline `{sample}.R3.fastq.gz`.


In [ ]:
prepared = []
prepared_map = {
    CDNA_SAMPLE: {
        'R1': FASTQ_SOURCE_FILES[CDNA_SAMPLE]['R1'],
        'R2': FASTQ_SOURCE_FILES[CDNA_SAMPLE]['R2'],
    },
    BEAD_SAMPLE: {
        'R1': FASTQ_SOURCE_FILES[BEAD_SAMPLE]['R1'],
        'R2': FASTQ_SOURCE_FILES[BEAD_SAMPLE]['R2'],
        'R3': FASTQ_SOURCE_FILES[BEAD_SAMPLE]['R3'],
    },
}

for sample, read_map in prepared_map.items():
    for read_name, src in read_map.items():
        src = Path(src).resolve()
        dst = FASTQ_DIR / f'{sample}.{read_name}.fastq.gz'
        if not src.exists():
            raise FileNotFoundError(src)
        if dst.exists() or dst.is_symlink():
            try:
                same_target = dst.resolve() == src
            except FileNotFoundError:
                same_target = False
            if same_target:
                status = 'exists'
            elif FORCE_RELINK_FASTQ:
                dst.unlink()
                os.symlink(src, dst)
                status = 'linked'
            else:
                raise FileExistsError(f'{dst} already exists and does not point to {src}. Set FORCE_RELINK_FASTQ=True to replace it.')
        else:
            os.symlink(src, dst)
            status = 'linked'
        prepared.append({'sample': sample, 'read': read_name, 'source': str(src), 'prepared': str(dst), 'status': status})

CDNA_SAMPLE_FILE = SAMPLE_SHEET_DIR / 'cDNA_samples.txt'
BEAD_SAMPLE_FILE = SAMPLE_SHEET_DIR / 'bead_samples.txt'
CDNA_SAMPLE_FILE.write_text(CDNA_SAMPLE + '\n')
BEAD_SAMPLE_FILE.write_text(BEAD_SAMPLE + '\n')

display(pd.DataFrame(prepared))
print('cDNA sample sheet:', CDNA_SAMPLE_FILE)
print('bead sample sheet:', BEAD_SAMPLE_FILE)


## 4. Import author scripts

The notebook imports the scripts directly from the GitHub clone. It does not edit those scripts; compatibility fixes are done in notebook state.


In [ ]:
EASYSPATIAL_DIR = SCRIPT_FROM_GITHUB / 'EasySpatial'
BEAD_SCRIPT_DIR = SCRIPT_FROM_GITHUB / 'Bead_interaction_pipeline'
for path in [str(EASYSPATIAL_DIR), str(BEAD_SCRIPT_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

Spatial_UMI_barcode_extraction = importlib.import_module('Spatial_UMI_barcode_extraction')
Fastq_trim_multi_files = importlib.import_module('Fastq_trim_multi_files')
STAR_module = importlib.import_module('STAR')
Sam_filter_multi_files = importlib.import_module('Sam_filter_multi_files')
Sam_rm_dup_barcode_UMI_multi_files = importlib.import_module('Sam_rm_dup_barcode_UMI_multi_files')
Sam_gene_counting_multi_files = importlib.import_module('Sam_gene_counting_multi_files')
Summary_gene_count_multi_files = importlib.import_module('Summary_gene_count_multi_files')
Generate_adata = importlib.import_module('Generate_adata')
Count_reads = importlib.import_module('Count_reads')

Bead_UMI_barcode_extraction = importlib.import_module('UMI_barcode_extraction')
Bead_spatial_barcode_extraction = importlib.import_module('spatial_barcode_extraction')
Bead_Remove_duplicate_barcode = importlib.import_module('Remove_duplicate_barcode')

# Author bead spatial script reads this global variable internally.
Bead_spatial_barcode_extraction.list_barcode_4_file2 = str(BARCODE_4_FILE)

print('Imported author scripts from:', SCRIPT_FROM_GITHUB)


## 5. cDNA FASTQ to expression matrix

Author order: barcode/UMI attach -> polyA trimming -> STAR alignment -> SAM filtering -> barcode+UMI duplicate removal -> gene counting -> count summary -> AnnData generation.


In [ ]:
adata_full_path = ADATA_FOLDER / 'adata_full.h5ad'
core_cdna = 32
umi_limit_for_adata = 50

if RUN_CDNA_PIPELINE:
    if adata_full_path.exists() and not FORCE_RERUN_CDNA:
        print('Skipping cDNA pipeline because output exists:', adata_full_path)
    else:
        Spatial_UMI_barcode_extraction.extract_spatial_barcode_files(
            str(FASTQ_DIR), str(CDNA_SAMPLE_FILE), str(FASTQ_BARCODE_ATTACHED), core_cdna,
            str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_BEAD1_FILE),
            mismatch_rate=1,
        )
        Fastq_trim_multi_files.Fastq_trim_files(
            str(FASTQ_BARCODE_ATTACHED), str(CDNA_SAMPLE_FILE), str(FASTQ_TRIMMED), core_cdna,
            adapter_seq='AAAAAAAA', cutadapt_path=str(CUTADAPT),
        )
        STAR_module.Fastq_star_alignment_multi_files(
            str(FASTQ_TRIMMED), str(CDNA_SAMPLE_FILE), str(SAM_STAR), core_cdna,
            index=str(STAR_INDEX), star_path=str(STAR),
        )
        Sam_filter_multi_files.Sam_filter_files(
            str(SAM_STAR), str(CDNA_SAMPLE_FILE), str(SAM_FILTERED), core_cdna,
            samtools_path=str(SAMTOOLS),
        )
        Sam_rm_dup_barcode_UMI_multi_files.rm_dup_files(
            str(SAM_FILTERED), str(CDNA_SAMPLE_FILE), str(SAM_RMDUP), core_cdna,
        )
        Sam_gene_counting_multi_files.scRNA_count_parallel(
            str(SAM_RMDUP), str(CDNA_SAMPLE_FILE), str(BED_GENE_COUNT), str(GENE_REFERENCE_PICKLE), core_cdna,
        )
        Summary_gene_count_multi_files.Gene_count_summary(
            str(BED_GENE_COUNT), str(CDNA_SAMPLE_FILE), str(SUMMARY_GENE_COUNT),
        )
        Generate_adata.generate_adata_from_gene_count(
            str(SUMMARY_GENE_COUNT), str(BED_GENE_COUNT / 'gene_anno.csv'), str(ADATA_FOLDER), umi_limit_for_adata,
        )
        print('Saved:', adata_full_path)
else:
    print('RUN_CDNA_PIPELINE=False; cDNA FASTQ processing was skipped.')


### cDNA read-count report


In [ ]:
cdna_report_path = REPORT_DIR / 'cDNA_read_number_summary.csv'
core_cdna = 32
if adata_full_path.exists():
    cdna_report = pd.DataFrame(index=[CDNA_SAMPLE])
    try:
        cdna_report['STAR_input_reads'] = Count_reads.Fastq_count_reads_files(str(FASTQ_BARCODE_ATTACHED) + '/', str(CDNA_SAMPLE_FILE), core=core_cdna)
    except Exception as exc:
        cdna_report['STAR_input_reads'] = np.nan
        print('Could not count barcode-attached FASTQ reads:', repr(exc))
    try:
        star_counts = Count_reads.Count_Align_STAR_files(str(SAM_STAR), str(CDNA_SAMPLE_FILE), core=core_cdna)
        star_counts.columns = ['STAR_input', 'STAR_unique', 'STAR_multi'][:star_counts.shape[1]]
        cdna_report = cdna_report.join(star_counts, how='left')
    except Exception as exc:
        print('Could not parse STAR final logs:', repr(exc))
    try:
        cdna_report['filtered_sam_reads'] = Count_reads.SAM_count_reads_files(str(SAM_FILTERED), str(CDNA_SAMPLE_FILE), core=core_cdna)
        cdna_report['dedup_sam_reads'] = Count_reads.SAM_count_reads_files(str(SAM_RMDUP), str(CDNA_SAMPLE_FILE), core=core_cdna)
        cdna_report['gene_assigned_reads'] = Count_reads.count_mapped_reads_files(str(BED_GENE_COUNT), str(CDNA_SAMPLE_FILE), core=core_cdna)
    except Exception as exc:
        print('Could not count SAM/gene-assigned reads:', repr(exc))
    cdna_report.to_csv(cdna_report_path)
    display(cdna_report)
    print('Saved:', cdna_report_path)
else:
    print('cDNA AnnData does not exist yet; run the cDNA pipeline first.')


### construct cDNA adata_leiden

In [ ]:
adata_full = sc.read_h5ad("/p2/zulab/jtian/data/IRISeq/second/output/my-second-pipeline/new/cDNA/Adata/adata_full.h5ad")

In [ ]:
adata_full.var["mt"] = adata_full.var["Gene_name"].str.lower().str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata_full, qc_vars=["mt"], inplace=True)

In [ ]:
g = sc.pl.violin(
    adata_full,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.2,
    size=2,
    multi_panel=True,
    show=False
)

g.axes.flat[0].set_ylim(0, 1000)  
g.axes.flat[1].set_ylim(0, 2000)  
g.axes.flat[2].set_ylim(0, 40)
plt.show()

In [ ]:
adata_full = adata_full[
    (adata_full.obs["total_counts"] >= 100) &
    (adata_full.obs["n_genes_by_counts"] >= 80) &
    (adata_full.obs["pct_counts_mt"] < 15),
    :
].copy()

In [ ]:
g = sc.pl.violin(
    adata_full,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.2,
    size=2,
    multi_panel=True,
    show=False
)

g.axes.flat[0].set_ylim(0, 1000)  
g.axes.flat[1].set_ylim(0, 2000)  
g.axes.flat[2].set_ylim(0, 40)
plt.show()

In [ ]:
adata_full_raw = adata_full.copy()

sc.pp.normalize_total(adata_full, target_sum=1e4)
sc.pp.log1p(adata_full)

sc.pp.highly_variable_genes(
    adata_full,
    min_mean=0.0125,
    max_mean=5,
    min_disp=0.5,
)

sc.pl.highly_variable_genes(adata_full)

adata_full = adata_full[:, adata_full.var["highly_variable"]].copy()

sc.pp.regress_out(
    adata_full,
    ['total_counts', 'pct_counts_mt']
)

sc.pp.scale(adata_full, max_value=10)

sc.tl.pca(adata_full)
sc.pl.pca_variance_ratio(adata_full, log=True)

sc.pp.neighbors(
    adata_full,
    n_neighbors=20,
    n_pcs=30
)

sc.tl.umap(
    adata_full,
    min_dist=0,
    spread=3
)
sc.tl.leiden(
    adata_full,
    resolution=1.7
)
sc.pl.umap(
    adata_full,
    color=['leiden']
)


In [ ]:
adata_full.write_h5ad('/p2/zulab/jtian/data/IRISeq/second/output/my-second-beads/adata_leiden_filter.h5ad')  

## Bead-interaction FASTQ to deduplicated receiver-sender connections



In [ ]:
spatial_rmdup_path = DEDUPLICATE_SPATIAL / f'{BEAD_SAMPLE}.spatial.csv.gz'
core_beads = 32

if RUN_BEAD_PIPELINE:
    if spatial_rmdup_path.exists() and not FORCE_RERUN_BEADS:
        print('Skipping bead pipeline because output exists:', spatial_rmdup_path)
    else:
        umi_attach_fastq = UMI_ATTACH / f'{BEAD_SAMPLE}.R2.fastq.gz'
        if umi_attach_fastq.exists() and not FORCE_RERUN_BEADS:
            print('Reusing existing UMI-attach FASTQ:', umi_attach_fastq)
        else:
            Bead_UMI_barcode_extraction.extract_spatial_barcode_files(
                str(FASTQ_DIR), str(BEAD_SAMPLE_FILE), str(UMI_ATTACH), core_beads,
                str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_BEAD1_FILE),
                mismatch_rate=1,
            )

        # Repair author interaction UMI-attach output when UMI newline splits the header.
        tmp = umi_attach_fastq.with_suffix(umi_attach_fastq.suffix + '.tmp_repaired')
        repaired_records = 0
        standard_records = 0
        with gzip.open(umi_attach_fastq, 'rt') as src_handle, gzip.open(tmp, 'wt', compresslevel=1) as dst_handle:
            while True:
                line1 = src_handle.readline()
                if not line1:
                    break
                line2 = src_handle.readline()
                if not line2:
                    raise ValueError(f'Truncated UMI-attach FASTQ after header: {umi_attach_fastq}')
                if line1.startswith('@') and line2.startswith(','):
                    seq = src_handle.readline()
                    plus = src_handle.readline()
                    qual = src_handle.readline()
                    if not qual:
                        raise ValueError(f'Truncated five-line UMI-attach record in {umi_attach_fastq}')
                    dst_handle.write(line1.rstrip('\n') + line2)
                    dst_handle.write(seq)
                    dst_handle.write(plus)
                    dst_handle.write(qual)
                    repaired_records += 1
                else:
                    seq = line2
                    plus = src_handle.readline()
                    qual = src_handle.readline()
                    if not qual:
                        raise ValueError(f'Truncated four-line UMI-attach record in {umi_attach_fastq}')
                    dst_handle.write(line1)
                    dst_handle.write(seq)
                    dst_handle.write(plus)
                    dst_handle.write(qual)
                    standard_records += 1
        os.replace(tmp, umi_attach_fastq)
        print(f'Repaired {umi_attach_fastq}: merged {repaired_records} five-line records; kept {standard_records} standard records.')

        bad_headers = []
        with gzip.open(umi_attach_fastq, 'rt') as handle:
            for idx in range(1000):
                header = handle.readline()
                if not header:
                    break
                seq = handle.readline()
                plus = handle.readline()
                qual = handle.readline()
                if not (header.startswith('@') and plus.startswith('+') and len(header.strip().split(',')) >= 5):
                    bad_headers.append((idx + 1, header.strip(), plus.strip()))
                    break
        if bad_headers:
            raise ValueError(f'UMI-attach FASTQ header validation failed: {bad_headers[:1]}')

        Bead_spatial_barcode_extraction.list_barcode_4_file2 = str(BARCODE_4_FILE)
        Bead_spatial_barcode_extraction.extract_spatial_barcode_files(
            str(UMI_ATTACH), str(BEAD_SAMPLE_FILE), str(SPATIAL_BARCODE), core_beads,
            str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_FILE),
        )
        Bead_Remove_duplicate_barcode.remove_duplicates_files(
            str(SPATIAL_BARCODE), str(BEAD_SAMPLE_FILE), str(DEDUPLICATE_SPATIAL), core_beads,
        )
        print('Saved:', spatial_rmdup_path)
else:
    print('RUN_BEAD_PIPELINE=False; bead FASTQ processing was skipped.')


### Bead interaction read-count report


In [ ]:
if spatial_rmdup_path.exists():
    bead_report_rows = []
    raw_r2 = FASTQ_DIR / f'{BEAD_SAMPLE}.R2.fastq.gz'
    umi_r2 = UMI_ATTACH / f'{BEAD_SAMPLE}.R2.fastq.gz'
    spatial_txt = SPATIAL_BARCODE / f'{BEAD_SAMPLE}.spatial.txt.gz'

    if raw_r2.exists():
        with gzip.open(raw_r2, 'rt') as handle:
            raw_r2_lines = sum(1 for _ in handle)
    else:
        raw_r2_lines = np.nan
    if umi_r2.exists():
        with gzip.open(umi_r2, 'rt') as handle:
            umi_r2_lines = sum(1 for _ in handle)
    else:
        umi_r2_lines = np.nan
    if spatial_txt.exists():
        with gzip.open(spatial_txt, 'rt') as handle:
            spatial_txt_lines = sum(1 for _ in handle)
    else:
        spatial_txt_lines = np.nan
    if spatial_rmdup_path.exists():
        with gzip.open(spatial_rmdup_path, 'rt') as handle:
            spatial_rmdup_lines = sum(1 for _ in handle)
    else:
        spatial_rmdup_lines = np.nan

    bead_report_rows.append({
        'sample': BEAD_SAMPLE,
        'total_reads': raw_r2_lines // 4 if pd.notna(raw_r2_lines) else np.nan,
        'Filtering_bead1_barcode': umi_r2_lines // 4 if pd.notna(umi_r2_lines) else np.nan,
        'Filtering_bead2_barcode': spatial_txt_lines,
        'Remove_duplicates_file_lines': spatial_rmdup_lines,
    })
    bead_read_number = pd.DataFrame(bead_report_rows)
    bead_read_number_path = BEAD_REPORT_FOLDER / 'read_number.csv'
    bead_read_number.to_csv(bead_read_number_path, index=False)
    display(bead_read_number)
    print('Saved:', bead_read_number_path)
else:
    print('Deduplicated spatial connection file does not exist yet.')


## Spatial reconstruction from bead-bead interactions


In [ ]:
#读入connection matrix
connection_dir = Path("/p2/zulab/jtian/data/IRISeq/second/output/my-second-pipeline/old/connections")

matrix_sparse = load_npz(connection_dir / "receiver_sender_connection_matrix_min_umi.npz")

receiver_barcodes = pd.read_csv(connection_dir / "receiver_barcodes.csv")["receiver_barcode"]
sender_barcodes = pd.read_csv(connection_dir / "sender_barcodes.csv")["sender_barcode"]

matrix_data_filter = pd.DataFrame(
    matrix_sparse.toarray(),
    index=receiver_barcodes.to_numpy(),
    columns=sender_barcodes.to_numpy(),
)

matrix_data_filter.index.name = "receiver_barcode"
matrix_data_filter.columns.name = "sender_barcode"


In [ ]:

# Log1p transformation
matrix_data_log1p = np.log1p(matrix_data_filter)

# Standardize the data (optional but often recommended)
scaler = StandardScaler()
matrix_data_standardized = scaler.fit_transform(matrix_data_log1p)

# Perform PCA on the standardized and log-transformed data
pca = PCA(n_components=6000)
pca.fit(matrix_data_standardized)

# Plot the explained variance as a function of the number of components
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Explained Variance')
plt.title('Elbow Plot for PCA')
plt.show()



In [ ]:
pca = PCA(n_components=4500)
matrix_data_pca = pca.fit_transform(matrix_data_standardized)


In [ ]:
# Set random seed for reproducibility
SEED = 42  # Set this to your desired seed value


# Create and fit the UMAP transformer
mapper = cuml.UMAP(
    n_neighbors=20,
    min_dist=0.75,
    metric='cosine',
    random_state=SEED,
    n_epochs=3000,
    learning_rate=1.0,
    verbose=True
)
coords = mapper.fit_transform(matrix_data_pca)

In [ ]:

def plot_umap(embedding, title='UMAP Embedding', s=1, figsize=(10, 10)):
    """
    Plots UMAP embeddings.
    
    Parameters:
    - embedding: UMAP embeddings (typically from mapper.fit_transform)
    - title: Title of the plot
    - s: Size of each point
    - figsize: Size of the figure
    """
    plt.figure(figsize=figsize)
    plt.scatter(embedding[:, 0], embedding[:, 1], s=s, cmap='Spectral')
    plt.gca().set_aspect('equal', 'datalim')
    plt.colorbar(boundaries=np.arange(11)-0.5).set_ticks(np.arange(10))
    plt.title(title)
    plt.show()

# To plot your coords
plot_umap(coords, title="UMAP projection with specified parameters")

In [ ]:
umap_df = pd.DataFrame(coords, columns=['UMAP1','UMAP2'], index=matrix_data_filter.index)